# AI Customer Support Agent: Performance Evaluation & Error Analysis
This notebook evaluates the end-to-end performance of the Twitter customer support agent across:
1. **Intent Classification Accuracy & Confusion Matrix**
2. **Escalation Triage: Precision, Recall, False Positives vs. Critical False Negatives**
3. **Retrieval Grounding & Top-k Match Performance**
4. **LLM-as-a-Judge Quality Metrics across 5 Evaluation Dimensions**

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
print("Libraries loaded successfully.")

## 1. Load Evaluation Benchmark Results

In [ ]:
pred_path = "../results/predictions.csv"
if not os.path.exists(pred_path):
    pred_path = "results/predictions.csv"
df_preds = pd.read_csv(pred_path)
print(f"Loaded {len(df_preds)} evaluation predictions.")
display(df_preds.head(3))

## 2. Intent Classification Performance

In [ ]:
intent_acc = (df_preds["gold_intent"] == df_preds["pred_intent"]).mean()
print(f"Intent Classification Accuracy: {intent_acc*100:.2f}%")

cm_path = "../results/confusion_matrix.json"
if not os.path.exists(cm_path):
    cm_path = "results/confusion_matrix.json"
with open(cm_path, "r") as f:
    cm_data = json.load(f)

labels = [l.replace("_", "
") for l in cm_data["labels"]]
plt.figure(figsize=(10, 8))
sns.heatmap(cm_data["matrix"], annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("Intent Classification Confusion Matrix", fontsize=13, fontweight="bold")
plt.xlabel("Predicted Intent")
plt.ylabel("Gold Intent")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 3. Escalation Triage Trade-off Analysis

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
esc_cr = classification_report(df_preds["gold_escalation"], df_preds["pred_escalation"], output_dict=True)
print("Escalation Classification Report:")
display(pd.DataFrame(esc_cr).transpose())

# False Negative vs False Positive Inspection
missed_risks = df_preds[(df_preds["gold_escalation"] == "ESCALATE_TO_HUMAN") & (df_preds["pred_escalation"] == "AUTO_HANDLE")]
unnecessary_esc = df_preds[(df_preds["gold_escalation"] == "AUTO_HANDLE") & (df_preds["pred_escalation"] == "ESCALATE_TO_HUMAN")]
print(f"Critical Missed Escalations (False Negatives): {len(missed_risks)}")
print(f"Unnecessary Escalations (False Positives):    {len(unnecessary_esc)}")

## 4. LLM-as-a-Judge Response Quality Distribution

In [ ]:
judge_cols = [c for c in df_preds.columns if c.startswith("judge_") and c != "judge_verdict"]
if judge_cols:
    mean_scores = df_preds[judge_cols].mean()
    plt.figure(figsize=(9, 4))
    mean_scores.plot(kind="bar", color="#8b5cf6")
    plt.title("Average Quality Scores Across Evaluation Rubrics (1 to 5)", fontsize=12, fontweight="bold")
    plt.ylabel("Score (1-5)")
    plt.ylim(0, 5.5)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
    display(mean_scores)